In [1]:
!pip install -q transformers datasets jiwer torchaudio sentencepiece torchao --upgrade

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/autolyrics/results', exist_ok=True)
os.makedirs('/content/drive/MyDrive/autolyrics/checkpoints', exist_ok=True)

from datasets import load_dataset, Audio

# Load English songs only
dataset = load_dataset("jamendolyrics/jamendolyrics", "en", split="test")
print(f"Number of English songs: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Number of English songs: 20
Columns: ['name', 'url', 'artist', 'title', 'genre', 'license_type', 'language', 'lyric_overlap', 'polyphonic', 'non_lexical', 'text', 'lines', 'words', 'audio']


In [3]:
sample = dataset[0]

print("=== SONG INFO ===")
print(f"Title: {sample['title']}")
print(f"Artist: {sample['artist']}")
print(f"Language: {sample['language']}")
print(f"Genre: {sample['genre']}")

print("\n=== AUDIO INFO ===")
print(f"Sampling rate: {sample['audio']['sampling_rate']}")
print(f"Audio array shape: {sample['audio']['array'].shape}")
duration = len(sample['audio']['array']) / sample['audio']['sampling_rate']
print(f"Duration: {duration:.1f} seconds")

print("\n=== LYRICS (first 200 chars) ===")
print(sample['text'][:200])

print("\n=== FIRST 5 WORDS WITH TIMESTAMPS ===")
for word in sample['words'][:5]:
    print(word)

print("\n=== FIRST 3 LINES WITH TIMESTAMPS ===")
for line in sample['lines'][:3]:
    print(line)

=== SONG INFO ===
Title: Give Me The Same
Artist: HILA
Language: en
Genre: Pop

=== AUDIO INFO ===
Sampling rate: 44100
Audio array shape: (10302862,)
Duration: 233.6 seconds

=== LYRICS (first 200 chars) ===
lay awake at night
wondering how could i
let it get this way
through all the pain
i took all the blame
while you cursed my last name
i thought we could get better
stayed committed like a soldier
belie

=== FIRST 5 WORDS WITH TIMESTAMPS ===
{'start': 18.6199798584, 'end': 18.8935203552, 'text': 'lay', 'line_end': False}
{'start': 18.8935203552, 'end': 19.2129116058, 'text': 'awake', 'line_end': False}
{'start': 19.2129116058, 'end': 19.4400577545, 'text': 'at', 'line_end': False}
{'start': 19.4400577545, 'end': 19.8730163574, 'text': 'night', 'line_end': True}
{'start': 20.8442592621, 'end': 21.1843261719, 'text': 'wondering', 'line_end': False}

=== FIRST 3 LINES WITH TIMESTAMPS ===
{'start': 18.6199798584, 'end': 19.8730163574, 'text': 'lay awake at night'}
{'start': 20.844259262

In [4]:
# Whisper strictly requires 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Verify
sample = dataset[0]
print(f"New sampling rate: {sample['audio']['sampling_rate']}")
print(f"New audio shape: {sample['audio']['array'].shape}")
duration = len(sample['audio']['array']) / 16000
print(f"Duration: {duration:.1f} seconds")
# Print all song titles so you can manually decide splits
print("All 20 English songs:")
for i, song in enumerate(dataset):
    duration = len(song['audio']['array']) / 16000
    print(f"  [{i}] {song['title']} by {song['artist']} ({duration:.0f}s)")
# Manually assign indices after reviewing the list above
# Adjust these indices based on your printed list
train_indices = [0,1,2,3,4,5,6,7,8,9,10,11,12,13]
val_indices   = [14,15,16]
test_indices  = [17,18,19]

train_songs = dataset.select(train_indices)
val_songs   = dataset.select(val_indices)
test_songs  = dataset.select(test_indices)

print(f"Train: {len(train_songs)} songs")
print(f"Val:   {len(val_songs)} songs")
print(f"Test:  {len(test_songs)} songs")

# Save split info to Drive for reproducibility
with open('/content/drive/MyDrive/autolyrics/results/splits.txt', 'w') as f:
    f.write(f"Train indices: {train_indices}\n")
    f.write(f"Val indices: {val_indices}\n")
    f.write(f"Test indices: {test_indices}\n")
    for split_name, split in [("TRAIN", train_songs), ("VAL", val_songs), ("TEST", test_songs)]:
        f.write(f"\n{split_name}:\n")
        for song in split:
            f.write(f"  {song['title']} - {song['artist']}\n")
print("Splits saved to Drive")

New sampling rate: 16000
New audio shape: (3738000,)
Duration: 233.6 seconds
All 20 English songs:
  [0] Give Me The Same by HILA (234s)
  [1] Keep On by Quentin Hannappe (190s)
  [2] Back In Time by Songwriterz (209s)
  [3] Peyote by KINEMATIC (183s)
  [4] Embers by Avercage (243s)
  [5] Falling Star by Colour Out (222s)
  [6] One Way Street by The.Madpix.Project (171s)
  [7] The statement by Wordsmith (171s)
  [8] Crowdpleaser by Jason Miller (193s)
  [9] Is It Right? by Lower Loveday (180s)
  [10] The Leader by Pure Mids (262s)
  [11] Fire Inside by Ridgway (284s)
  [12] Wrong Concept by Moon I mean (220s)
  [13] Whistler by Slingshot miracle (229s)
  [14] Voices by The RINN (231s)
  [15] Bad Side by Rxbyn (217s)
  [16] The One (feat Tina G) by Tom Orlando (240s)
  [17] Vision by LUNABLIND (221s)
  [18] Like The Sun by Explosive ear candy (193s)
  [19] Feel (Stripped) by Cortez (241s)
Train: 14 songs
Val:   3 songs
Test:  3 songs
Splits saved to Drive


In [5]:
import numpy as np

CHUNK_DURATION = 30      # seconds
SAMPLE_RATE    = 16000
CHUNK_SAMPLES  = CHUNK_DURATION * SAMPLE_RATE   # = 480000 samples

def chunk_song(song):
    """
    Splits one song into 30-second audio chunks,
    each paired with its ground truth lyrics.
    Returns a list of dicts.
    """
    audio_array = song['audio']['array']        # numpy array, 16kHz
    words       = song['words']                 # list of {start, end, text, line_end}
    total_duration = len(audio_array) / SAMPLE_RATE

    chunks = []
    chunk_start_time = 0.0

    while chunk_start_time < total_duration:
        chunk_end_time = chunk_start_time + CHUNK_DURATION

        # Slice audio
        start_sample = int(chunk_start_time * SAMPLE_RATE)
        end_sample   = min(int(chunk_end_time * SAMPLE_RATE), len(audio_array))
        audio_chunk  = audio_array[start_sample:end_sample]

        # Skip very short chunks at the end (less than 5 seconds)
        if len(audio_chunk) < 5 * SAMPLE_RATE:
            chunk_start_time = chunk_end_time
            continue

        # Find words that fall within this chunk
        chunk_words = [
            w['text'] for w in words
            if w['start'] >= chunk_start_time and w['start'] < chunk_end_time
        ]

        # Skip chunks with no lyrics (instrumental sections)
        if not chunk_words:
            chunk_start_time = chunk_end_time
            continue

        ground_truth = ' '.join(chunk_words).strip().lower()

        chunks.append({
            'audio_array':   audio_chunk,
            'ground_truth':  ground_truth,
            'song_title':    song['title'],
            'chunk_start':   chunk_start_time,
            'chunk_end':     chunk_end_time,
        })

        chunk_start_time = chunk_end_time

    return chunks


# Test on one song first
test_chunks = chunk_song(dataset[0])
print(f"Song: {dataset[0]['title']}")
print(f"Number of chunks: {len(test_chunks)}")
print(f"\nFirst chunk:")
print(f"  Audio shape: {test_chunks[0]['audio_array'].shape}")
print(f"  Duration: {len(test_chunks[0]['audio_array'])/SAMPLE_RATE:.1f}s")
print(f"  Ground truth: {test_chunks[0]['ground_truth']}")

Song: Give Me The Same
Number of chunks: 8

First chunk:
  Audio shape: (480000,)
  Duration: 30.0s
  Ground truth: lay awake at night wondering how could i let it get this way through all the pain i took all


In [6]:
def chunk_all_songs(song_split):
    all_chunks = []
    for song in song_split:
        chunks = chunk_song(song)
        all_chunks.extend(chunks)
        print(f"  {song['title']}: {len(chunks)} chunks")
    return all_chunks

print("Chunking train songs...")
train_chunks = chunk_all_songs(train_songs)

print("\nChunking val songs...")
val_chunks = chunk_all_songs(val_songs)

print("\nChunking test songs...")
test_chunks = chunk_all_songs(test_songs)

print(f"\nTotal chunks — Train: {len(train_chunks)}, Val: {len(val_chunks)}, Test: {len(test_chunks)}")

Chunking train songs...
  Give Me The Same: 8 chunks
  Keep On: 6 chunks
  Back In Time: 7 chunks
  Peyote: 5 chunks
  Embers: 7 chunks
  Falling Star: 7 chunks
  One Way Street: 6 chunks
  The statement: 6 chunks
  Crowdpleaser: 6 chunks
  Is It Right?: 6 chunks
  The Leader: 6 chunks
  Fire Inside: 9 chunks
  Wrong Concept: 7 chunks
  Whistler: 8 chunks

Chunking val songs...
  Voices: 7 chunks
  Bad Side: 7 chunks
  The One (feat Tina G): 7 chunks

Chunking test songs...
  Vision: 8 chunks
  Like The Sun: 6 chunks
  Feel (Stripped): 8 chunks

Total chunks — Train: 94, Val: 21, Test: 22


In [7]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
from jiwer import wer, cer

# Load vanilla whisper — no modifications
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model     = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)
print(f"Running on: {device}")

def transcribe_chunk(audio_array):
    inputs = processor(
        audio_array,
        sampling_rate=16000,
        return_tensors="pt"
    )
    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]
    return transcription.strip().lower()


# Run on test chunks
print("Running baseline on test set...")
references  = []
hypotheses  = []

for i, chunk in enumerate(test_chunks):
    pred = transcribe_chunk(chunk['audio_array'])
    references.append(chunk['ground_truth'])
    hypotheses.append(pred)

    # Print first 5 to see qualitative performance
    if i < 5:
        print(f"\n[Chunk {i+1}]")
        print(f"  REF : {chunk['ground_truth']}")
        print(f"  PRED: {pred}")

# Compute metrics
baseline_wer = wer(references, hypotheses)
baseline_cer = cer(references, hypotheses)

print(f"\n=== BASELINE RESULTS ===")
print(f"WER: {baseline_wer:.4f} ({baseline_wer*100:.1f}%)")
print(f"CER: {baseline_cer:.4f} ({baseline_cer*100:.1f}%)")

# Save to Drive — this number is sacred
with open('/content/drive/MyDrive/autolyrics/results/baseline_wer.txt', 'w') as f:
    f.write(f"Baseline WER: {baseline_wer:.4f}\n")
    f.write(f"Baseline CER: {baseline_cer:.4f}\n")
    f.write(f"Test chunks evaluated: {len(test_chunks)}\n")

print("\nBaseline saved to Drive.")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Running on: cuda
Running baseline on test set...


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform


[Chunk 1]
  REF : racking and jacking my mind up cannot be telling the time i don't care no i don't care
  PRED: i

[Chunk 2]
  REF : no you send thirty messages drunk and dirty messages drunk but i don't look no i can't be to look i could not breathe you took your toll on me you're suffocating please but since i watched you leave i can see for the first time
  PRED: send 30 messages drunk and dirty messages drunk but i don't look no i can't be put to luck i cannot breathe you took your toll on me you're suffocating please but since i watched you leave i can't see for the first time

[Chunk 3]
  REF : and i can see for the first time and i can see for the first time for the first time and i can see for the first time you get jealous easily
  PRED: and i can't see for the first time and i can't see for the first time for the first time and i can't see for the first time you get gendered easily

[Chunk 4]
  REF : you did not make it easy but what was i expecting it's a very fitting endi

In [8]:
#!pip install -q transformers==4.36.2 accelerate==0.25.0 peft==0.7.1
#!pip install -q "transformers==4.40.0" "accelerate==0.28.0" "peft==0.10.0"
!pip install -q "accelerate>=1.1.0" "transformers>=4.40.0" "peft>=0.10.0"

In [9]:
import torch
import numpy as np
from datasets import Dataset
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small")

def preprocess_chunk(chunk):
    """Convert raw audio chunk + ground truth into model inputs"""
    # Audio → mel spectrogram features
    inputs = processor(
        chunk['audio_array'],
        sampling_rate=16000,
        return_tensors="pt"
    )

    # Ground truth text → token ids (labels)
    labels = processor.tokenizer(
        chunk['ground_truth'],
        return_tensors="pt"
    ).input_ids

    return {
        "input_features": inputs.input_features.squeeze(0),  # shape: (80, 3000)
        "labels": labels.squeeze(0)                           # shape: (seq_len,)
    }

def build_hf_dataset(chunks):
    processed = [preprocess_chunk(c) for c in chunks]
    return processed  # list of dicts

print("Preprocessing train chunks...")
train_processed = build_hf_dataset(train_chunks)

print("Preprocessing val chunks...")
val_processed = build_hf_dataset(val_chunks)

print("Preprocessing test chunks...")
test_processed = build_hf_dataset(test_chunks)

print(f"Done. Train: {len(train_processed)}, Val: {len(val_processed)}, Test: {len(test_processed)}")
print(f"\nSample input_features shape: {train_processed[0]['input_features'].shape}")
print(f"Sample labels shape: {train_processed[0]['labels'].shape}")

Preprocessing train chunks...
Preprocessing val chunks...
Preprocessing test chunks...
Done. Train: 94, Val: 21, Test: 22

Sample input_features shape: torch.Size([80, 3000])
Sample labels shape: torch.Size([23])


In [10]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Input features are all same shape (80, 3000) — just stack them
        input_features = torch.stack([f["input_features"] for f in features])

        # Labels need padding since they vary in length
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        # Replace padding token id with -100 so loss ignores padding
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove decoder start token if present (Whisper adds it automatically)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        return {
            "input_features": input_features,
            "labels": labels
        }

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator ready")

Data collator ready


In [11]:
from jiwer import wer
import numpy as np

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 back to pad token so we can decode
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions and labels to text
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Lowercase both before computing WER
    pred_str  = [p.lower().strip() for p in pred_str]
    label_str = [l.lower().strip() for l in label_str]

    wer_score = wer(label_str, pred_str)

    return {"wer": wer_score}

print("Metrics function ready")

Metrics function ready


In [13]:
# Run this first to see exact layer names
for name, module in model.named_modules():
    if "attn" in name and ("q_proj" in name or "v_proj" in name):
        print(name)

model.encoder.layers.0.self_attn.v_proj
model.encoder.layers.0.self_attn.q_proj
model.encoder.layers.1.self_attn.v_proj
model.encoder.layers.1.self_attn.q_proj
model.encoder.layers.2.self_attn.v_proj
model.encoder.layers.2.self_attn.q_proj
model.encoder.layers.3.self_attn.v_proj
model.encoder.layers.3.self_attn.q_proj
model.encoder.layers.4.self_attn.v_proj
model.encoder.layers.4.self_attn.q_proj
model.encoder.layers.5.self_attn.v_proj
model.encoder.layers.5.self_attn.q_proj
model.encoder.layers.6.self_attn.v_proj
model.encoder.layers.6.self_attn.q_proj
model.encoder.layers.7.self_attn.v_proj
model.encoder.layers.7.self_attn.q_proj
model.encoder.layers.8.self_attn.v_proj
model.encoder.layers.8.self_attn.q_proj
model.encoder.layers.9.self_attn.v_proj
model.encoder.layers.9.self_attn.q_proj
model.encoder.layers.10.self_attn.v_proj
model.encoder.layers.10.self_attn.q_proj
model.encoder.layers.11.self_attn.v_proj
model.encoder.layers.11.self_attn.q_proj
model.decoder.layers.0.self_attn.v_p

In [14]:
from transformers import WhisperForConditionalGeneration
from peft import get_peft_model, LoraConfig, TaskType

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.use_cache = False
model.generation_config.language = "english"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

# Decoder only — 12 decoder layers
target_modules_decoder = []
for i in range(12):
    target_modules_decoder.extend([
        f"model.decoder.layers.{i}.self_attn.q_proj",
        f"model.decoder.layers.{i}.self_attn.v_proj",
        f"model.decoder.layers.{i}.encoder_attn.q_proj",
        f"model.decoder.layers.{i}.encoder_attn.v_proj",
    ])

lora_config_decoder = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=target_modules_decoder,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model_exp2 = get_peft_model(model, lora_config_decoder)
model_exp2.print_trainable_parameters()

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 589,824 || all params: 242,324,736 || trainable%: 0.24340230788490366


In [15]:
from transformers import Seq2SeqTrainingArguments

# Carefully tuned for your dataset size (94 train chunks)
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/autolyrics/checkpoints/exp2_decoder",

    # Batch & steps
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch size = 16

    # Steps — with 94 samples and batch 16, one epoch ≈ 6 steps
    # 300 steps ≈ ~50 epochs — right amount for this dataset size
    max_steps=300,
    warmup_steps=30,

    # Learning rate — higher than full fine-tune since LoRA is small
    learning_rate=1e-4,
    lr_scheduler_type="linear",

    # Memory
    fp16=True,
    gradient_checkpointing=True,

    # Evaluation
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,                 # only keep 2 best checkpoints to save Drive space
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,            # lower WER is better

    # Generation during eval
    predict_with_generate=True,
    generation_max_length=128,

    logging_steps=25,
    report_to="none",                   # switch to "wandb" if you want live charts
    push_to_hub=False,
)

print("Training args ready")

Training args ready


In [16]:
import torch
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
model_exp2 = model_exp2.to(device)

SAVE_DIR = "/content/drive/MyDrive/autolyrics/checkpoints/exp2_decoder"
os.makedirs(SAVE_DIR, exist_ok=True)

def collate_fn(batch):
    input_features = torch.stack([torch.tensor(b["input_features"]) for b in batch])
    labels = [torch.tensor(b["labels"]) for b in batch]
    max_len = max(l.shape[0] for l in labels)
    padded_labels = torch.full((len(labels), max_len), -100, dtype=torch.long)
    for i, l in enumerate(labels):
        padded_labels[i, :l.shape[0]] = l
    return {"input_features": input_features, "labels": padded_labels}

train_loader = DataLoader(train_processed, batch_size=8, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_processed,   batch_size=8, shuffle=False, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(
    model_exp2.parameters(),
    lr=3e-5,           # much lower LR — key fix
    weight_decay=0.01  # regularization to prevent overfitting
)

MAX_STEPS    = 100    # much fewer steps for small dataset
WARMUP_STEPS = 10
EVAL_EVERY   = 20

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=MAX_STEPS
)

step = 0
best_val_loss = float("inf")
patience = 0
MAX_PATIENCE = 3      # early stopping — stop if val loss doesn't improve 3 evals in a row
train_iter = iter(train_loader)

print("Starting Experiment 2: LoRA Decoder Only (Fixed)...")
print("="*50)

while step < MAX_STEPS:
    model_exp2.train()

    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    input_features = batch["input_features"].to(device)
    labels         = batch["labels"].to(device)

    with torch.cuda.amp.autocast():   # mixed precision
        outputs = model_exp2.base_model.model(
            input_features=input_features,
            labels=labels
        )

    loss = outputs.loss
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model_exp2.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    step += 1

    if step % 10 == 0:
        print(f"Step {step}/{MAX_STEPS} | Train Loss: {loss.item():.4f}")

    if step % EVAL_EVERY == 0:
        model_exp2.eval()
        val_loss_total = 0
        val_steps = 0

        with torch.no_grad():
            for val_batch in val_loader:
                val_features = val_batch["input_features"].to(device)
                val_labels   = val_batch["labels"].to(device)
                with torch.cuda.amp.autocast():
                    val_outputs = model_exp2.base_model.model(
                        input_features=val_features,
                        labels=val_labels
                    )
                val_loss_total += val_outputs.loss.item()
                val_steps += 1

        avg_val_loss = val_loss_total / val_steps
        print(f"  → Val Loss at step {step}: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience = 0
            model_exp2.save_pretrained(f"{SAVE_DIR}/best")
            processor.save_pretrained(f"{SAVE_DIR}/best")
            print(f"  → New best saved (val loss: {best_val_loss:.4f})")
        else:
            patience += 1
            print(f"  → No improvement. Patience: {patience}/{MAX_PATIENCE}")
            if patience >= MAX_PATIENCE:
                print("  → Early stopping triggered.")
                break

print("\nTraining complete!")

# Load best checkpoint before evaluating
from peft import PeftModel
model_base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model_exp2 = PeftModel.from_pretrained(model_base, f"{SAVE_DIR}/best")
model_exp2 = model_exp2.to(device)
model_exp2.eval()
print("Best checkpoint loaded for evaluation")

Starting Experiment 2: LoRA Decoder Only (Fixed)...


/tmp/ipykernel_18380/568713634.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_features = torch.stack([torch.tensor(b["input_features"]) for b in batch])
/tmp/ipykernel_18380/568713634.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(b["labels"]) for b in batch]
/tmp/ipykernel_18380/568713634.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():   # mixed precision


Step 10/100 | Train Loss: 4.0638
Step 20/100 | Train Loss: 2.3345


/tmp/ipykernel_18380/568713634.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  → Val Loss at step 20: 2.1118
  → New best saved (val loss: 2.1118)
Step 30/100 | Train Loss: 2.0955
Step 40/100 | Train Loss: 2.4554
  → Val Loss at step 40: 1.8213
  → New best saved (val loss: 1.8213)
Step 50/100 | Train Loss: 1.9113
Step 60/100 | Train Loss: 1.6025
  → Val Loss at step 60: 1.7293
  → New best saved (val loss: 1.7293)
Step 70/100 | Train Loss: 1.7154
Step 80/100 | Train Loss: 2.2932
  → Val Loss at step 80: 1.6913
  → New best saved (val loss: 1.6913)
Step 90/100 | Train Loss: 2.0153
Step 100/100 | Train Loss: 1.7042
  → Val Loss at step 100: 1.6803
  → New best saved (val loss: 1.6803)

Training complete!


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Best checkpoint loaded for evaluation


In [34]:
import torch
from jiwer import wer, cer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_exp2.eval()
model_exp2 = model_exp2.to(device)

print("Evaluating Experiment 2 on test set...")
references = []
hypotheses = []

for i, chunk in enumerate(test_chunks):
    inputs = processor(
        chunk['audio_array'],
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model_exp2.base_model.model.generate(inputs)

    pred = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    references.append(chunk['ground_truth'].lower().strip())
    hypotheses.append(pred.lower().strip())

    if i < 5:
        print(f"\n[Chunk {i+1}]")
        print(f"  REF : {chunk['ground_truth']}")
        print(f"  PRED: {pred.lower()}")
import re

def normalize(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    text = text.replace("30", "thirty")
    text = text.replace("&", "and")
    return text

# Normalize both before WER
references = [normalize(r) for r in references]
hypotheses = [normalize(h) for h in hypotheses]

# Now compute WER on normalized text
exp2_wer = wer(references, hypotheses)  # or exp3_wer
exp2_cer = cer(references, hypotheses)
baseline_wer = 0.3040
exp2_relative = (baseline_wer - exp2_wer) / baseline_wer * 100

print(f"\n=== EXPERIMENT 2 RESULTS ===")
print(f"WER: {exp2_wer:.4f} ({exp2_wer*100:.1f}%)")
print(f"CER: {exp2_cer:.4f} ({exp2_cer*100:.1f}%)")
print(f"Relative WER reduction: {exp2_relative:.1f}%")

with open('/content/drive/MyDrive/autolyrics/results/exp2_results.txt', 'w') as f:
    f.write(f"Experiment 2: LoRA Decoder Only\n")
    f.write(f"WER: {exp2_wer:.4f}\n")
    f.write(f"CER: {exp2_cer:.4f}\n")
    f.write(f"Relative WER reduction: {exp2_relative:.1f}%\n")

print("Results saved to Drive")

Evaluating Experiment 2 on test set...

[Chunk 1]
  REF : racking and jacking my mind up cannot be telling the time i don't care no i don't care
  PRED:  i

[Chunk 2]
  REF : no you send thirty messages drunk and dirty messages drunk but i don't look no i can't be to look i could not breathe you took your toll on me you're suffocating please but since i watched you leave i can see for the first time
  PRED:  send 30 messages drunk and dirty messages drunk but i don't look no i can't be put to luck i cannot breathe you took your toll on me you're suffocating please but since i watched you leave i can't see for the first time

[Chunk 3]
  REF : and i can see for the first time and i can see for the first time for the first time and i can see for the first time you get jealous easily
  PRED:  and i can't see for the first time and i can't see for the first time for the first time and i can't see for the first time you get gendered easily

[Chunk 4]
  REF : you did not make it easy but wha

In [28]:
from transformers import WhisperForConditionalGeneration
from peft import get_peft_model, LoraConfig, TaskType

model2 = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model2.config.use_cache = False
model2.generation_config.language = "english"
model2.generation_config.task = "transcribe"
model2.generation_config.forced_decoder_ids = None

target_modules_both = []
for i in range(12):
    target_modules_both.extend([
        f"model.decoder.layers.{i}.self_attn.q_proj",
        f"model.decoder.layers.{i}.self_attn.v_proj",
        f"model.decoder.layers.{i}.encoder_attn.q_proj",
        f"model.decoder.layers.{i}.encoder_attn.v_proj",
    ])
for i in range(12):
    target_modules_both.extend([
        f"model.encoder.layers.{i}.self_attn.q_proj",
        f"model.encoder.layers.{i}.self_attn.v_proj",
    ])

lora_config_both = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=target_modules_both,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model_exp3 = get_peft_model(model2, lora_config_both)
model_exp3.print_trainable_parameters()

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 884,736 || all params: 242,619,648 || trainable%: 0.36465966680489126


In [29]:
import torch
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
model_exp3 = model_exp3.to(device)

SAVE_DIR = "/content/drive/MyDrive/autolyrics/checkpoints/exp3_both"
os.makedirs(SAVE_DIR, exist_ok=True)

train_loader = DataLoader(train_processed, batch_size=8, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_processed,   batch_size=8, shuffle=False, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(
    model_exp3.parameters(),
    lr=3e-5,
    weight_decay=0.01
)

MAX_STEPS    = 100
WARMUP_STEPS = 10
EVAL_EVERY   = 20
MAX_PATIENCE = 3

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=MAX_STEPS
)

step = 0
best_val_loss = float("inf")
patience = 0
train_iter = iter(train_loader)

print("Starting Experiment 3: LoRA Encoder + Decoder...")
print("="*50)

while step < MAX_STEPS:
    model_exp3.train()

    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    input_features = batch["input_features"].to(device)
    labels         = batch["labels"].to(device)

    with torch.cuda.amp.autocast():
        outputs = model_exp3.base_model.model(
            input_features=input_features,
            labels=labels
        )

    loss = outputs.loss
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model_exp3.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    step += 1

    if step % 10 == 0:
        print(f"Step {step}/{MAX_STEPS} | Train Loss: {loss.item():.4f}")

    if step % EVAL_EVERY == 0:
        model_exp3.eval()
        val_loss_total = 0
        val_steps = 0

        with torch.no_grad():
            for val_batch in val_loader:
                val_features = val_batch["input_features"].to(device)
                val_labels   = val_batch["labels"].to(device)
                with torch.cuda.amp.autocast():
                    val_outputs = model_exp3.base_model.model(
                        input_features=val_features,
                        labels=val_labels
                    )
                val_loss_total += val_outputs.loss.item()
                val_steps += 1

        avg_val_loss = val_loss_total / val_steps
        print(f"  → Val Loss at step {step}: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience = 0
            model_exp3.save_pretrained(f"{SAVE_DIR}/best")
            processor.save_pretrained(f"{SAVE_DIR}/best")
            print(f"  → New best saved (val loss: {best_val_loss:.4f})")
        else:
            patience += 1
            print(f"  → No improvement. Patience: {patience}/{MAX_PATIENCE}")
            if patience >= MAX_PATIENCE:
                print("  → Early stopping triggered.")
                break

print("\nTraining complete!")

# Load best checkpoint
from peft import PeftModel
from transformers import WhisperForConditionalGeneration
model_base3 = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model_exp3  = PeftModel.from_pretrained(model_base3, f"{SAVE_DIR}/best")
model_exp3  = model_exp3.to(device)
model_exp3.eval()
print("Best checkpoint loaded for evaluation")

Starting Experiment 3: LoRA Encoder + Decoder...


/tmp/ipykernel_18380/568713634.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_features = torch.stack([torch.tensor(b["input_features"]) for b in batch])
/tmp/ipykernel_18380/568713634.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(b["labels"]) for b in batch]
/tmp/ipykernel_18380/3892595354.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Step 10/100 | Train Loss: 3.1355
Step 20/100 | Train Loss: 2.4286


/tmp/ipykernel_18380/3892595354.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  → Val Loss at step 20: 2.0788
  → New best saved (val loss: 2.0788)
Step 30/100 | Train Loss: 2.0008
Step 40/100 | Train Loss: 2.0816
  → Val Loss at step 40: 1.7912
  → New best saved (val loss: 1.7912)
Step 50/100 | Train Loss: 2.0163
Step 60/100 | Train Loss: 2.3352
  → Val Loss at step 60: 1.7102
  → New best saved (val loss: 1.7102)
Step 70/100 | Train Loss: 1.7308
Step 80/100 | Train Loss: 1.9447
  → Val Loss at step 80: 1.6710
  → New best saved (val loss: 1.6710)
Step 90/100 | Train Loss: 2.0071
Step 100/100 | Train Loss: 1.8502
  → Val Loss at step 100: 1.6590
  → New best saved (val loss: 1.6590)

Training complete!


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Best checkpoint loaded for evaluation


In [33]:
model_exp3.eval()
model_exp3 = model_exp3.to(device)

print("Evaluating Experiment 3 on test set...")
references = []
hypotheses = []

for i, chunk in enumerate(test_chunks):
    inputs = processor(
        chunk['audio_array'],
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model_exp3.base_model.model.generate(inputs)

    pred = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    references.append(chunk['ground_truth'].lower().strip())
    hypotheses.append(pred.lower().strip())

    if i < 5:
        print(f"\n[Chunk {i+1}]")
        print(f"  REF : {chunk['ground_truth']}")
        print(f"  PRED: {pred.lower()}")
import re

def normalize(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    text = text.replace("30", "thirty")
    text = text.replace("&", "and")
    return text

# Normalize both before WER
references = [normalize(r) for r in references]
hypotheses = [normalize(h) for h in hypotheses]

# Now compute WER on normalized text
exp3_wer = wer(references, hypotheses)  # or exp3_wer
exp3_cer = cer(references, hypotheses)
exp3_relative = (baseline_wer - exp3_wer) / baseline_wer * 100

print(f"\n=== EXPERIMENT 3 RESULTS ===")
print(f"WER: {exp3_wer:.4f} ({exp3_wer*100:.1f}%)")
print(f"CER: {exp3_cer:.4f} ({exp3_cer*100:.1f}%)")
print(f"Relative WER reduction: {exp3_relative:.1f}%")

with open('/content/drive/MyDrive/autolyrics/results/exp3_results.txt', 'w') as f:
    f.write(f"Experiment 3: LoRA Encoder + Decoder\n")
    f.write(f"WER: {exp3_wer:.4f}\n")
    f.write(f"CER: {exp3_cer:.4f}\n")
    f.write(f"Relative WER reduction: {exp3_relative:.1f}%\n")

print("Results saved to Drive")

Evaluating Experiment 3 on test set...

[Chunk 1]
  REF : racking and jacking my mind up cannot be telling the time i don't care no i don't care
  PRED:  i

[Chunk 2]
  REF : no you send thirty messages drunk and dirty messages drunk but i don't look no i can't be to look i could not breathe you took your toll on me you're suffocating please but since i watched you leave i can see for the first time
  PRED:  send 30 messages drunk and dirty messages drunk but i don't look no i can't be put to luck i cannot breathe you took your toll on me you're suffocating please but since i watched you leave i can't see for the first time

[Chunk 3]
  REF : and i can see for the first time and i can see for the first time for the first time and i can see for the first time you get jealous easily
  PRED:  and i can't see for the first time and i can't see for the first time for the first time and i can't see for the first time you get gendered easily

[Chunk 4]
  REF : you did not make it easy but wha

In [32]:
print(f"DEBUG: I think exp2_wer is {exp2_wer} and exp3_wer is {exp3_wer}")

DEBUG: I think exp2_wer is 0.20814977973568283 and exp3_wer is 0.20814977973568283


In [35]:
baseline_cer = 0.2120

print("\n" + "="*65)
print("FINAL RESULTS COMPARISON")
print("="*65)
print(f"{'Experiment':<30} {'WER':>8} {'CER':>8} {'Rel. WER↓':>12}")
print("-"*65)
print(f"{'Baseline (zero-shot)':<30} {baseline_wer:>8.4f} {baseline_cer:>8.4f} {'—':>12}")
print(f"{'Exp2: LoRA Decoder Only':<30} {exp2_wer:>8.4f} {exp2_cer:>8.4f} {(baseline_wer-exp2_wer)/baseline_wer*100:>11.1f}%")
print(f"{'Exp3: LoRA Enc+Dec':<30} {exp3_wer:>8.4f} {exp3_cer:>8.4f} {exp3_relative:>11.1f}%")
print("="*65)

with open('/content/drive/MyDrive/autolyrics/results/final_results.txt', 'w') as f:
    f.write("FINAL RESULTS\n")
    f.write("="*65 + "\n")
    f.write(f"Baseline  — WER: {baseline_wer:.4f}, CER: {baseline_cer:.4f}\n")
    f.write(f"Exp2      — WER: {exp2_wer:.4f}, CER: {exp2_cer:.4f}, Rel WER reduction: {(baseline_wer-exp2_wer)/baseline_wer*100:.1f}%\n")
    f.write(f"Exp3      — WER: {exp3_wer:.4f}, CER: {exp3_cer:.4f}, Rel WER reduction: {exp3_relative:.1f}%\n")

print("\nAll results saved to Drive")


FINAL RESULTS COMPARISON
Experiment                          WER      CER    Rel. WER↓
-----------------------------------------------------------------
Baseline (zero-shot)             0.3040   0.2120            —
Exp2: LoRA Decoder Only          0.2192   0.1623        27.9%
Exp3: LoRA Enc+Dec               0.2081   0.1463        31.5%

All results saved to Drive
